
### Low-ℓ BB — Full-Bundle Coadd Tile Viewer


#### Workflow
 --------
 1. Set paths and configuration (§1)
 2. Load coadd T / Q / U from FITS (§2)
 3. Load available apodisation + point-source masks 
 4. Inspect field extent from observed pixels 
 5. Build tile grid 
 6. Tile loop — **unmasked** first, to see raw point sources 
 7. Tile loop — **with mask applied** 

In [1]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
from spt3g import core, maps

In [3]:
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)

from Plot import (
    apply_spt_style,
    mask_map,
    show_map_full_field,
    show_map_thumbnail,
)

apply_spt_style()

In [ ]:
# Coadd path 
DATA_DIR   = "/sptgrid/analysis/spt3g_d1_midell_tqu_healpix"  

# print the content of the data directory to verify the path
print("Contents of data directory:")
for item in os.listdir(DATA_DIR):
    print(item)


COADD_FILE  = "real_data_maps/full"   
COADD_PATH = os.path.join(DATA_DIR, COADD_FILE)

# Print the name of the coadd fits inside the path,
print("\nCoadd files in the coadd path:")
for item in os.listdir(COADD_PATH):
        print(item)

full_220ghz = COADD_PATH + "/full_220ghz.fits"
full_150ghz = COADD_PATH + "/full_150ghz.fits"
full_095ghz  = COADD_PATH + "/full_095ghz.fits"


# ── Mask directory ────────────────────────────────────────────────────────────
MASK_DIR = "/sptlocal/user/creichardt/bb2020"

# Print the content of the mask directory 
print("\nContents of mask directory:")
for item in os.listdir(MASK_DIR):
    print(item)




Contents of data directory:
README.html
simulated_maps
ancillary_products
real_data_maps

Coadd files in the coadd path:
full_220ghz.fits
full_150ghz.fits
full_095ghz.fits

Contents of mask directory:
puremask_0p5medwt_100mJy_30arcmin.npz
c2mask_0p5medwt_120arcmin.npz
circletest_puremask_0p5medwt_radius480_240arcmin.npz
circletest_c2puremask_0p5medwt_radius960_120arcmin.npz
puremask_0p5medwt_250mJy_180arcmin.npz
circletest_puremask_0p5medwt_radius960_480arcmin.npz
border_lines_0p5medwt.npz
circletest_puremask_0p5medwt_radius1800_240arcmin.npz
circletest_c2puremask_0p5medwt_radius2400_240arcmin.npz
distance_xy_0p5medwt.npz
distance_0p5medwt.npz
mask_badpixels_180arcmin.npz
circletest_c2puremask_0p5medwt_radius960_240arcmin.npz
puremask_0p5medwt_60arcmin.npz
circletest_puremask_0p5medwt_radius480_120arcmin.npz
bk8192_250mJy_nodisk20.npz
circletest_c2puremask_0p5medwt_radius2400_120arcmin.npz
circletest_puremask_0p5medwt_radius2400_480arcmin.npz
distance8192_0p6deg_xy_0p5medwt.npz
circlet

In [16]:
MASK_FILES = {
    "mask_250_30"   : "puremask8192_0p5medwt_250mJy_30arcmin.npz",
    "mask_250_60"   : "puremask8192_0p5medwt_250mJy_60arcmin.npz",
    "mask_250_nd30" : "puremask8192_0p5medwt_250mJy_nodisk_30arcmin.npz",
    "mask_250_nd60" : "puremask8192_0p5medwt_250mJy_nodisk_60arcmin.npz",
    "mask_100_30"   : "puremask8192_0p5medwt_100mJy_30arcmin.npz",
    "mask_apod_30"  : "puremask8192_0p5medwt_30arcmin.npz",
    "mask_apod_60"  : "puremask8192_0p5medwt_60arcmin.npz",
}

In [17]:
# display parameters 
RESO_ARCMIN  = 0.2     # arcmin per pixel  
PATCH_DEG    = 3.0     # square patch width = height in degrees
PATCH_PIX    = int(PATCH_DEG * 60 / RESO_ARCMIN)   # = 900 px for 3° patches

STOKES_KEYS  = ["T", "Q", "U"]
CMAP         = "coolwarm"

# Set to a directory path to save PDFs instead of displaying inline.
# Leave as None to show figures in the notebook.
SAVE_DIR     = None    # e.g. "/path/to/output/tiles/"

print(f"Patch size  : {PATCH_DEG}° × {PATCH_DEG}°  =  {PATCH_PIX} × {PATCH_PIX} px")
print(f"Resolution  : {RESO_ARCMIN} arcmin/px")

Patch size  : 3.0° × 3.0°  =  900 × 900 px
Resolution  : 0.2 arcmin/px


In [21]:
# 220 ghz map
T_c, Q_c, U_c = hp.read_map(full_220ghz, field=(0, 1, 2), partial=False)

# Build the observed-pixel boolean mask

obs_mask = (
        np.isfinite(T_c) & (T_c != hp.UNSEEN) &
        np.isfinite(Q_c) & (Q_c != hp.UNSEEN) &
        np.isfinite(U_c) & (U_c != hp.UNSEEN)
    )

stokes_maps = {"T": T_c, "Q": Q_c, "U": U_c}
nside_c     = hp.get_nside(T_c)

print(f"nside            : {nside_c}")
print(f"Observed pixels  : {obs_mask.sum():,}  ({obs_mask.sum()/len(T_c)*100:.2f}% of sky)")

nside            : 8192
Observed pixels  : 36,557,509  (4.54% of sky)


In [23]:
# load one of the masks
MASK_PATH = os.path.join(MASK_DIR, "puremask8192_0p5medwt_250mJy_30arcmin.npz")
data      = np.load(MASK_PATH)
apod      = data[data.files[0]].astype(float)
print(f"Mask loaded, shape: {apod.shape}, key: '{data.files[0]}'")

Mask loaded, shape: (805306368,), key: 'mask'


In [24]:
obs_pix        = np.where(obs_mask)[0]
theta_c, phi_c = hp.pix2ang(nside_c, obs_pix)

ra_obs  = np.degrees(phi_c)
ra_obs  = np.where(ra_obs > 180, ra_obs - 360, ra_obs)   # −180 … +180 deg
dec_obs = 90.0 - np.degrees(theta_c)

ra_min,  ra_max  = ra_obs.min(),  ra_obs.max()
dec_min, dec_max = dec_obs.min(), dec_obs.max()

print(f"RA  range  :  {ra_min:.2f}° → {ra_max:.2f}°   span = {ra_max-ra_min:.2f}°")
print(f"Dec range  : {dec_min:.2f}° → {dec_max:.2f}°   span = {dec_max-dec_min:.2f}°")


RA  range  :  -56.51° → 53.76°   span = 110.28°
Dec range  : -71.98° → -40.20°   span = 31.78°


In [ ]:
# Quick full-field overview (optional sanity check)
m_T_full = mask_map(T_c, obs_mask.astype(float))
rms_T    = float(np.std(m_T_full[m_T_full != hp.UNSEEN]))
show_map_full_field(
    m_T_full, vmin=-3*rms_T, vmax=3*rms_T,
    title="Full-bundle coadd — T  (full field overview)",
    unit="Tcmb", cmap=CMAP,
)

In [ ]:
# %%
ra_centres  = np.arange(ra_min  + PATCH_DEG/2, ra_max  + PATCH_DEG/2, PATCH_DEG)
dec_centres = np.arange(dec_min + PATCH_DEG/2, dec_max + PATCH_DEG/2, PATCH_DEG)

n_ra, n_dec = len(ra_centres), len(dec_centres)
n_total     = n_ra * n_dec * len(STOKES_KEYS)

print(f"RA  centres  : {n_ra}  ({ra_centres[0]:.1f}° … {ra_centres[-1]:.1f}°)")
print(f"Dec centres  : {n_dec}  ({dec_centres[0]:.1f}° … {dec_centres[-1]:.1f}°)")
print(f"Patches/Stokes : {n_ra * n_dec}")
print(f"Total panels : {n_total}  ({len(STOKES_KEYS)} Stokes × {n_ra*n_dec} patches)")
if n_total > 200:
    print("  ⚠  Many panels — consider setting SAVE_DIR to write PDFs instead.")


In [ ]:
def _tile_loop(stokes_maps_in, obs_mask_in, label=""):
    """
    Iterate over all tiles and Stokes components, calling show_map_thumbnail
    for each patch (inline) or saving to SAVE_DIR (if set in §1).
    """
    if SAVE_DIR is not None:
        os.makedirs(SAVE_DIR, exist_ok=True)

    for stokes in STOKES_KEYS:
        arr = stokes_maps_in[stokes].copy()
        arr[~obs_mask_in] = hp.UNSEEN

        rms        = float(np.std(arr[obs_mask_in]))
        vmin, vmax = -3 * rms, 3 * rms

        print(f"  {stokes}  rms = {rms:.4g}  →  colour limits ±{3*rms:.4g}")

        for dec_c in dec_centres:
            for ra_c in ra_centres:
                title = (
                    f"{stokes}{('  ' + label) if label else ''}  |  "
                    f"RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°"
                )
                rot = (ra_c, dec_c, 0)

                if SAVE_DIR is not None:
                    # Save mode: drive gnomview directly so we control savefig
                    fig = plt.figure(figsize=(6, 6), facecolor="white")
                    hp.gnomview(
                        arr,
                        rot=rot,
                        xsize=PATCH_PIX, ysize=PATCH_PIX,
                        reso=RESO_ARCMIN,
                        cmap=CMAP, min=vmin, max=vmax,
                        badcolor="white",
                        title=title, unit="Tcmb",
                        fig=fig.number,
                    )
                    safe_label = label.replace(" ", "_").replace("/", "-")
                    fname = (
                        f"tile_{stokes}{('_' + safe_label) if safe_label else ''}"
                        f"_ra{ra_c:+.1f}_dec{dec_c:+.1f}.pdf"
                    )
                    plt.savefig(
                        os.path.join(SAVE_DIR, fname),
                        bbox_inches="tight", dpi=120,
                    )
                    plt.close(fig)
                else:
                    # Inline mode: use the existing square-thumbnail helper
                    show_map_thumbnail(
                        arr, vmin=vmin, vmax=vmax,
                        title=title, unit="Tcmb",
                        cmap=CMAP,
                        rot=rot,
                        xsize=PATCH_PIX, ysize=PATCH_PIX,
                        reso=RESO_ARCMIN,
                    )

In [ ]:
SAVE_DIR = "/sptlocal/user/vwelke/lowl_bb_tiles"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:




print("=== Unmasked tiles ===")
_tile_loop(stokes_maps, obs_mask, label="unmasked")

In [ ]:
ACTIVE_MASK = "mask_250_30"   # ← change to any key from masks dict

assert ACTIVE_MASK in masks, (
    f"'{ACTIVE_MASK}' not loaded. Available: {list(masks.keys())}"
)

apod = masks[ACTIVE_MASK]

# Sanity-check nside compatibility
nside_mask = hp.get_nside(apod)
if nside_mask != nside_c:
    print(f"Warning: mask nside={nside_mask}, map nside={nside_c} — upgranding mask.")
    apod = hp.ud_grade(apod, nside_out=nside_c)

# Combined observed mask: observed pixels AND mask > 0
obs_and_mask = obs_mask & (apod > 0)

# Apply apodisation: multiply map values by mask weight inside observed region
stokes_masked = {}
for stokes in STOKES_KEYS:
    arr = stokes_maps[stokes].copy()
    arr[obs_and_mask]  *= apod[obs_and_mask]   # taper at edges / source holes
    arr[~obs_and_mask]  = hp.UNSEEN
    stokes_masked[stokes] = arr

print(f"Active mask  : {ACTIVE_MASK}  ({MASK_FILES[ACTIVE_MASK]})")
print(f"Pixels after mask  : {obs_and_mask.sum():,}  "
      f"(was {obs_mask.sum():,} unmasked)")

# ──────────────────────────────────────────────────────────────────────────────
print(f"\n=== Masked tiles  [{ACTIVE_MASK}] ===")
_tile_loop(stokes_masked, obs_and_mask, label=ACTIVE_MASK)
